# Semana 13: K-Means y recomendación básica

Notebook de clase para segmentación, silhouette y similitud item-item.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
X, y_true = make_blobs(n_samples=180, centers=3, cluster_std=0.55, random_state=21)
print(X.shape)

In [ ]:
rows = []
models = {}
for k in [2, 3, 4, 5]:
    model = KMeans(n_clusters=k, n_init=10, random_state=21).fit(X)
    labels = model.labels_
    rows.append({"k": k, "inertia": model.inertia_, "silhouette": silhouette_score(X, labels)})
    models[k] = model
pd.DataFrame(rows)

In [ ]:
best = max(rows, key=lambda row: row["silhouette"])["k"]
model = models[best]
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X[:, 0], X[:, 1], c=model.labels_, s=18, cmap="viridis")
ax.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1], c="red", s=120, marker="x")
ax.set_title(f"K-Means con k={best}")
plt.close(fig)
fig

## Recomendación item-item

In [ ]:
items = ["calculo", "algebra", "python", "estadistica", "visualizacion"]
M = np.array([
    [5, 4, 0, 1, 0],
    [4, 5, 0, 1, 0],
    [0, 1, 5, 4, 4],
    [1, 0, 4, 5, 4],
    [0, 0, 4, 4, 5],
], dtype=float)
sim = cosine_similarity(M)
pd.DataFrame(sim, index=items, columns=items)

In [ ]:
def recommend(item_index, similarity, seen_items=None, top_k=2):
    seen = set() if seen_items is None else set(seen_items)
    seen.add(item_index)
    candidates = [idx for idx in range(similarity.shape[0]) if idx not in seen]
    candidates.sort(key=lambda idx: similarity[item_index, idx], reverse=True)
    return candidates[:top_k]

idx = items.index("python")
print([items[i] for i in recommend(idx, sim, seen_items={idx}, top_k=2)])

## Cierre

Un cluster o una recomendación es una hipótesis operacional: necesita evidencia, estabilidad y revisión de límites.